# xLSTM, sLSTM, mLSTMs - with Checkpoints

## Preparation

### Import modules

In [ ]:
# Cell 1: Mount & Navigate
from google.colab import drive

drive.mount('/content/drive')

# Go to correct folder
%cd /content/drive/MyDrive/Colab\ Notebooks/thesis/LSTM_Train

# Verify structure
!ls -la ../
# Should show: dataset/  thesis_utils/  LSTM_Train/

!pip install loguru torchxlstm fastparquet

import sys
from pathlib import Path
import os

# Add thesis_utils to path (parent dir)
sys.path.insert(0, '/content/drive/MyDrive/Colab\ Notebooks/thesis')

# Or simpler:
sys.path.insert(0, str(Path.cwd().parent))

In [27]:
# Prediction using LSTM, GRU-LSTM, xLSTM

# Prediction using LSTM, GRU-LSTM, xLSTM
import copy
import math
from typing import List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from pandas import DataFrame
from sklearn.model_selection import KFold, GroupShuffleSplit
from torch.nn.utils import clip_grad_norm_
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import DataLoader, Dataset, Subset

import thesis_utils as tu

Added to path: /Users/gabriel/Documents/01_TUW/00_Thesis/trade_under_pressure_thesis/src
thesis_utils exists: True


In [ ]:
CHKPT_DIR = "/content/drive/MyDrive/Colab\ Notebooks/thesis/checkpoints"
os.makedirs(CHKPT_DIR, exist_ok=True)

In [ ]:
def ckpt_path(serial, fold):
  return os.path.join(CHKPT_DIR, f"{serial}_fold{fold}.pt")

### Configuration

In [28]:
# Model parameters
HORIZON = 1
BATCH_SIZE = 128
EMBEDDING_SIZE = 128
NUM_EPOCHS = 60
HIDDEN_SIZE = 256
N_LAYERS = 3
DROPOUT = 0.05
XLSTM_TYPE = "X"
N_LAGS = 5

# Train parameters
TARGET = "EXPORT_centered"
FEATURES = [
  "contig", "comlang_off", "colony", "smctry",
]
N_SPLITS = 8
PATIENCE = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0
RANDOM_SEED = 16
SUBSAMPLE_ENABLED = False
N_DYADS = 10
XLSTM_LAYERS = "sm"

SANCTION_COLS = ["arms", "military", "trade", "travel", "other"]

# Torch config
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))

dyads_case_study = [
  # "USA_CHN", "CHN_USA",
  # "USA_CAN", "CAN_USA",
  # "DEU_CHN", "CHN_DEU",
  # "USA_DEU", "DEU_USA",
  # "USA_MEX", "MEX_USA",
  # "AUS_CHN", "CHN_AUS",
  # "USA_JPN", "JPN_USA",
  # "DEU_JPN", "JPN_DEU",
  # "USA_AUS", "AUS_USA",
  # "DEU_RUS", "RUS_DEU",
]

In [29]:
# Save config
SAVE_ENABLED = False
layers_string = f"({XLSTM_LAYERS})"
SERIAL_NUMBER = (
  f"{XLSTM_TYPE}LSTM"
  f"{layers_string if XLSTM_TYPE == 'X' else ''}"
  f"-{LEARNING_RATE}lr-{DROPOUT}d-{HIDDEN_SIZE}hs-{WEIGHT_DECAY}wd-{BATCH_SIZE}bs-{N_LAYERS}layers-{EMBEDDING_SIZE}es-kfolds{N_SPLITS}-hp"
)
SERIAL_NUMBER = SERIAL_NUMBER.replace(".", "_")
PATH_TO_FOLDER = ""

### Load Data

In [30]:
processed = pd.read_parquet(path="../dataset/processed.parquet", engine="fastparquet")
df: DataFrame = processed.copy(deep=True)

### Sort, shift and compute data

In [31]:
# Sort data by Report + Partner + Year
df["dyad_id"] = df["ISO3_reporter"] + "_" + df["ISO3_partner"]
df = df.sort_values(by=["dyad_id", "Year"], ignore_index=True)

In [32]:
# Remove case study dyad_pairs
mask_keep = ~np.isin(df["dyad_id"], dyads_case_study)
df = df.loc[mask_keep].reset_index(drop=True)

In [33]:
# Sanity check case study pairs
has_overlap = df["dyad_id"].isin(dyads_case_study).any()

if has_overlap:
  print("⚠️ Some case study dyads are present in the DataFrame.")
else:
  print("✅ No case study dyads found in the DataFrame.")

✅ No case study dyads found in the DataFrame.


In [34]:
if SUBSAMPLE_ENABLED:
  dyad_subsample = pd.Series(df["dyad_id"].unique()).sample(n=N_DYADS, random_state=RANDOM_SEED, replace=False)
  df = df[df["dyad_id"].isin(dyad_subsample)]

print(f"Unique dyads: {df["dyad_id"].nunique()}")

Unique dyads: 33672


In [35]:
df["sanction"] = (df[SANCTION_COLS]
                  .sum(axis=1)).astype(int)

### Coerce numerical values and convert dyad_id to categorical

In [36]:
num_cols = ["distw", "GDP_reporter", "GDP_partner", "sanction", "contig",
            "comlang_off", "colony", "smctry", "Year", ]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(float)
df = df.dropna(subset=num_cols)

In [37]:
df["Year"] = df["Year"].astype(int)
for col in ["dyad_id"]:
  df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()))

In [38]:
# Save EXPORT std and median to undo centering
EXPORT_STD = df["EXPORT"].std()
EXPORT_MEDIAN = df["EXPORT"].median()

### Center data

In [39]:
center_columns = ["distw", "GDP_reporter", "GDP_partner", "EXPORT"]
for col in center_columns:
  median = df[col].median()
  std_df = df[col].std()
  df[col + "_centered"] = (df[col] - median) / std_df
FEATURES += ["distw_centered"]

In [40]:
lag_cols = ["GDP_reporter_centered", "GDP_partner_centered", "sanction"]
for col in lag_cols:
  for index in range(1, N_LAGS + 1):
    df[f"{col}_lag{index}"] = df.groupby("dyad_id", observed=True)[col].shift(index)

In [41]:
df = df.dropna()

In [42]:
FEATURES += [f"{c}_lag{index}" for c in lag_cols for index in range(1, N_LAGS + 1)]

## Split data

In [43]:
# Embeddings
dyad_to_idx = { dyad: i for i, dyad in enumerate(df["dyad_id"].cat.categories) }
df["dyad_idx"] = df["dyad_id"].map(dyad_to_idx).astype(int)

In [44]:
# Split into Train, Validation and Test sets
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)

train_idx, test_idx = next(gss.split(df, groups=df["dyad_id"]))
test_df = df.iloc[test_idx]
train_df = df.iloc[train_idx]

train_idx, val_idx = next(gss.split(train_df, groups=train_df["dyad_id"]))
val_df = train_df.iloc[val_idx]
train_df = train_df.iloc[train_idx]

In [45]:
train_df.loc[:, FEATURES] = train_df.loc[:, FEATURES].astype(
  "float32",
  copy=False
)

# Train

## Define Fold and Epoch steps
_For reusability_

In [46]:
# Create KFold object
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

In [ ]:
from torch.amp import autocast


def epoch_step(
    model: nn.Module,
    optimizer: Optimizer,
    criterion: nn.Module,
    scheduler: LRScheduler,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: any,
    scaler: torch.amp.GradScaler,
) -> float:
  # =========================
  # TRAIN
  # =========================
  model.train()

  for X, y, di in train_loader:
    X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))

    optimizer.zero_grad(set_to_none=True)

    # --- AMP forward ---
    with autocast("cuda"):
      y_pred = model(X, di)

      if not torch.isfinite(y_pred).all():
        print("⚠️ NaN or Inf detected in y_pred — stopping here!")
        return float("inf")

      loss = criterion(y_pred, y)

    if not torch.isfinite(loss):
      print("⚠️ loss is NaN or Inf!")
      return float("inf")

    # --- AMP backward ---
    scaler.scale(loss).backward()

    # IMPORTANT: unscale before clipping
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), max_norm=1.0)

    scaler.step(optimizer)
    scaler.update()

    # OneCycleLR MUST step per batch
    scheduler.step()

  # =========================
  # VALIDATION
  # =========================
  model.eval()
  val_losses = []

  with torch.no_grad():
    for X, y, di in val_loader:
      X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))
      preds = model(X, di)
      val_losses.append(criterion(preds, y).item())

  val_rmse = math.sqrt(sum(val_losses) / len(val_losses))
  return val_rmse

In [ ]:
from torch.amp import GradScaler


# Define fold step
def fold_step(fold: int, train_idx: List, val_idx: List,
              dataset: Dataset, batch_size: int, num_epochs: int,
              model: nn.Module, device: any,
              optimizer: Optimizer, criterion: nn.Module, serial: str) -> (float, dict):
  train_loader = DataLoader(
    Subset(dataset, train_idx),
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    persistent_workers=True,
    pin_memory=True
  )

  val_loader = DataLoader(
    Subset(dataset, val_idx),
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    persistent_workers=True,
    prefetch_factor=2,
    pin_memory=True
  )

  scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=7e-4, epochs=NUM_EPOCHS, steps_per_epoch=len(train_loader),
    div_factor=25, final_div_factor=100, pct_start=0.3, anneal_strategy="cos", three_phase=False,
  )

  ckpt_file = ckpt_path(serial, fold)
  start_epoch = 0
  best_rmse = float("inf")
  best_state = copy.deepcopy(model.state_dict())

  if os.path.exists(ckpt_file):
    print(f"🔄 Resuming from checkpoint: {ckpt_file}")
    ckpt = torch.load(ckpt_file, map_location=device)

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler = ckpt["scaler_state"]

    start_epoch = ckpt["epoch"] + 1
    best_rmse = ckpt["best_rmse"]
    best_state = ckpt["best_state"]
  else:
    scaler = GradScaler("cuda")

  print(f"Start epoch train for fold {fold}")
  scaler = GradScaler("cuda")
  for epoch in range(start_epoch, num_epochs):
    val_rmse = epoch_step(
      model=model,
      optimizer=optimizer,
      criterion=criterion,
      scheduler=scheduler,
      train_loader=train_loader,
      val_loader=val_loader,
      device=device,
      scaler=scaler,
    )
    print(f"Epoch {epoch + 1:02d}/{num_epochs}  |  val RMSE: {val_rmse:.4f}")

    if val_rmse < best_rmse - 1e-4:
      best_rmse, patience_left = val_rmse, 10
      best_state = model.state_dict()

    torch.save({
      "epoch": epoch,
      "model_state": model.state_dict(),
      "optimizer_state": optimizer.state_dict(),
      "scheduler_state": scheduler.state_dict(),
      "scaler_state": scaler.state_dict(),
      "best_rmse": best_rmse,
      "best_state": best_state,
    }, ckpt_file)

  model.load_state_dict(best_state)
  model.eval()

  preds_centered, truth_centered = [], []
  preds_raw, truth_raw = [], []

  with torch.no_grad():
    for X, y, di in val_loader:
      X, di = map(lambda t: t.to(device, non_blocking=True), (X, di))
      out = model(X, di).cpu()

      preds_centered.append(out)
      truth_centered.append(y)

      preds_raw.append(out)
      truth_raw.append(y)

  preds_centered = torch.cat(preds_centered).numpy()
  truth_centered = torch.cat(truth_centered).numpy()

  preds_raw = torch.cat(preds_raw).numpy() * EXPORT_STD + EXPORT_MEDIAN
  truth_raw = torch.cat(truth_raw).numpy() * EXPORT_STD + EXPORT_MEDIAN

  rmse_centered = tu.rmse(truth_centered, preds_centered)
  mae_centered = tu.mae(truth_centered, preds_centered)
  rmae_centered = tu.rmae(truth_centered, preds_centered)
  pseudo_r2_centered = tu.pseudo_r2(truth_centered, preds_centered)

  print(
    f"Fold {fold} CENTERED  RMSE {rmse_centered:.4f} | MAE {mae_centered:.4f} | R² {pseudo_r2_centered:.4f} | RMAE {rmae_centered:.4f}")

  rmse_raw = tu.rmse(truth_raw, preds_raw)
  mae_raw = tu.mae(truth_raw, preds_raw)
  rmae_raw = tu.rmae(truth_raw, preds_raw)
  pseudo_r2_raw = tu.pseudo_r2(truth_raw, preds_raw)

  print(
    f"Fold {fold} RAW  RMSE {rmse_raw:.4f} | MAE {mae_raw:.4f} | R² {pseudo_r2_raw:.4f} | RMAE {rmae_raw:.4f}"
  )

  return (
    { "RMSE": rmse_centered, "MAE": mae_centered, "R2": pseudo_r2_centered, "RMAE": rmae_centered },
    { "RMSE": rmse_raw, "MAE": mae_raw, "R2": pseudo_r2_raw, "RMAE": rmae_raw },
    copy.deepcopy(best_state))

## Train Raw dataset

### Split dataset

In [49]:
# Convert df_scaled to pytorch Tensor
dataset, dyad_to_idx = tu.make_panel_datasets_dyad(
  data=df,
  features=FEATURES,
  target=TARGET,
  horizon=HORIZON,
)

In [50]:
# Create DataLoaders for the 3 sets
train_loader = DataLoader(
  Subset(dataset, train_idx),
  batch_size=BATCH_SIZE,
  shuffle=True,
  num_workers=2,
  persistent_workers=True,

  pin_memory=True
)

val_loader = DataLoader(
  Subset(dataset, val_idx),
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=2,
  persistent_workers=True,

  pin_memory=True
)

test_loader = DataLoader(
  Subset(dataset, test_idx),
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=2,
  persistent_workers=True,

  pin_memory=True
)

### Train model

In [51]:
# Save best train iteration
best_fold_state = None
best_fold_rmse = float("inf")

metrics_per_fold = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

metrics_per_fold_raw = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

In [ ]:
for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(dataset))), 1):

  ckpt_file = ckpt_path(SERIAL_NUMBER, fold)
  if os.path.exists(ckpt_file):
    ckpt = torch.load(ckpt_file, map_location="cpu")
    if ckpt["epoch"] >= NUM_EPOCHS - 1:
      print(f"✅ Fold {fold} already completed — skipping.")
      continue

  print(f"=== FOLD {fold}/{N_SPLITS} ===")

  model = tu.DyadXLSTM(
    n_features=len(FEATURES),
    n_dyads=len(dyad_to_idx),
    embed_dim=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    horizon=HORIZON,
    type=XLSTM_TYPE,
    layers=XLSTM_LAYERS,
    n_layers=N_LAYERS,
  ).to(device=device)

  criterion = nn.SmoothL1Loss(beta=0.5)
  optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98), eps=1e-8)

  fold_metrics, fold_metrics_raw, best_state = fold_step(fold=fold,
                                                         train_idx=train_idx,
                                                         val_idx=val_idx,
                                                         dataset=dataset,
                                                         batch_size=BATCH_SIZE,
                                                         num_epochs=NUM_EPOCHS,
                                                         model=model,
                                                         device=device,
                                                         optimizer=optimizer,
                                                         criterion=criterion,
                                                         serial=SERIAL_NUMBER)
  # scheduler=scheduler)
  if fold_metrics["RMSE"] < best_fold_rmse:
    best_fold_rmse = fold_metrics["RMSE"]
    best_fold_state = copy.deepcopy(best_state)

  for k, v in fold_metrics.items():
    metrics_per_fold[k].append(v)

  for k, v in fold_metrics_raw.items():
    metrics_per_fold_raw[k].append(v)

=== FOLD 1/8 ===
Start epoch train for fold 1


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x31d88fe20>
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/thesis_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/thesis_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1622, in _shutdown_workers
    self._mark_worker_as_unavailable(worker_id, shutdown=True)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/thesis_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1558, in _mark_worker_as_unavailable
    assert self._workers_status[worker_id] or (
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


Epoch 01/60  |  val RMSE: 0.2146
Epoch 02/60  |  val RMSE: 0.1845
Epoch 03/60  |  val RMSE: 0.1714
Epoch 04/60  |  val RMSE: 0.1680
Epoch 05/60  |  val RMSE: 0.1643
Epoch 06/60  |  val RMSE: 0.1624
Epoch 07/60  |  val RMSE: 0.1685
Epoch 08/60  |  val RMSE: 0.1630
Epoch 09/60  |  val RMSE: 0.1602
Epoch 10/60  |  val RMSE: 0.1558
Epoch 11/60  |  val RMSE: 0.1604
Epoch 12/60  |  val RMSE: 0.1575
Epoch 13/60  |  val RMSE: 0.1511
Epoch 14/60  |  val RMSE: 0.1606
Epoch 15/60  |  val RMSE: 0.1513
Epoch 16/60  |  val RMSE: 0.1438
Epoch 17/60  |  val RMSE: 0.1476
Epoch 18/60  |  val RMSE: 0.1500
Epoch 19/60  |  val RMSE: 0.1393
Epoch 20/60  |  val RMSE: 0.1454
Epoch 21/60  |  val RMSE: 0.1435
Epoch 22/60  |  val RMSE: 0.1365
Epoch 23/60  |  val RMSE: 0.1394
Epoch 24/60  |  val RMSE: 0.1365
Epoch 25/60  |  val RMSE: 0.1348
Epoch 26/60  |  val RMSE: 0.1328
Epoch 27/60  |  val RMSE: 0.1313
Epoch 28/60  |  val RMSE: 0.1334
Epoch 29/60  |  val RMSE: 0.1275
Epoch 30/60  |  val RMSE: 0.1311
Epoch 31/6

## Save Model

In [ ]:
torch.save({
  "model_state_dict": best_fold_state,
  "model_hyperparams": {
    "n_features": len(FEATURES),
    "n_dyads": len(dyad_to_idx),
    "n_layers": N_LAYERS,
    "embed_dim": EMBEDDING_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "dropout": DROPOUT,
    "horizon": HORIZON,
    "type": XLSTM_TYPE,
    "layers": XLSTM_LAYERS,
  },
  "dyad_to_idx": dyad_to_idx,
  "feature_names": FEATURES,
}, PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")
print("Saved model to", PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")

In [ ]:
def summarize(xs):
  xs = np.asarray(xs, dtype=float)
  n = xs.size
  mean = xs.mean()
  std = xs.std(ddof=1)  # sample std
  se = std / math.sqrt(n)
  try:
    from scipy.stats import t
    tcrit = t.ppf(0.975, df=n - 1)
  except Exception:
    tcrit = 1.96  # normal approx≈
  ci95 = tcrit * se
  return mean, std, ci95

In [ ]:
print("\n=== Cross-fold CENTERED summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")

print("\n=== Cross-fold RAW summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold_raw[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")